# Single-task spatial overfitting ablation on Modal

This paired diagnostic asks whether the complete FLUX transformer can learn
the color-permutation rule in ARC task `0d3d703e` from 512 unique,
one-pass augmentations when only the final target is supervised. NVARC and
the FLUX VAE remain frozen.

Two independent runs start from the same verified FLUX basis checkpoint:

1. `compact-3x3`: a 48x48 target image and 9 FLUX image tokens, with no PAD;
2. `padded-30x30`: the primary 480x480 canvas and 900 image tokens.

Both runs use the same 512 augmentation keys, target order, optimizer
settings, and per-step timestep seeds. Evaluation at steps 0, 16, 32, and
64 covers eight seen views, eight disjoint unseen views, and the canonical
view with two generation seeds. The final evaluation additionally repeats
generation with deterministically permuted conditioning and identical noise.

Prerequisites:

- one Modal H200;
- the writable `diffusion-arc-models` Volume mounted at
  `/mnt/diffusion-arc-models`;
- the model and complete ARC-AGI-2 setup artifacts already in that Volume;
- a `GITHUB_TOKEN` Modal Secret;
- `DIFFUSION_ARC_REVISION` set to the exact pushed commit containing this
  experiment (the default `main` is convenient but not immutable).

Run top-to-bottom. The long experiment cell performs both runs sequentially,
uploads both validated checkpoints, and writes one comparison report.


In [1]:
!nvidia-smi


Tue Aug 25 20:36:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:97:00.0 Off |                    0 |
| N/A   33C    P0             77W /  700W |       4MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
%uv pip uninstall torchvision


Using Python 3.12.6 environment at: /usr/local
Uninstalled 1 package in 53ms
 - torchvision==0.23.0+cu129
Note: you may need to restart the kernel to use updated packages.


## 1. Immutable run configuration

The run ID is shared by both variants. The runner refuses incompatible
hardware, data counts, target counts, spatial shapes, or source contracts
before the first optimizer update.


In [8]:
from __future__ import annotations

import base64
from datetime import datetime, timezone
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import time

VOLUME_NAME = 'diffusion-arc-models'
VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
SOURCE_REVISION = "a2f33795cd64ca63514e8ebc9f9b79daee7c5e06"
AUTO_SHUTDOWN = os.environ.get('DIFFUSION_ARC_AUTO_SHUTDOWN', '1') == '1'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RESULTS_ROOT = VOLUME_ROOT / 'results' / 'single-task-spatial-overfitting-ablation'
REPORT_PATH = RESULTS_ROOT / RUN_ID / 'report.json'

if not SOURCE_REVISION:
    raise ValueError('DIFFUSION_ARC_REVISION must not be empty')
required = (
    VOLUME_ROOT / 'flux2-klein-base-4b',
    VOLUME_ROOT / 'nvarc-qwen3-4b-grids15-sft139',
    VOLUME_ROOT / 'download-manifest.json',
    VOLUME_ROOT / 'data' / 'raw' / 'arc-agi-2',
    VOLUME_ROOT / 'data' / 'arc-agi-2-manifest.json',
)
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required Volume artifacts: {missing}')

print('Run ID:', RUN_ID)
print('Requested source revision:', SOURCE_REVISION)
print('Final report:', REPORT_PATH)


Run ID: 20260825T203819809220Z
Requested source revision: a2f33795cd64ca63514e8ebc9f9b79daee7c5e06
Final report: /mnt/diffusion-arc-models/results/single-task-spatial-overfitting-ablation/20260825T203819809220Z/report.json


## 2. Fetch, install, and test the exact source

This cell resolves the requested revision, installs only mismatched pinned
packages, installs Diffusion-ARC editable from that checkout, and requires
the complete local unit-test suite to pass.


In [9]:
def run_command(label: str, command: list[str], *, cwd: Path | None = None, env=None) -> str:
    completed = subprocess.run(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-50:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()

SOURCE_DIR = Path('/tmp') / f'diffusion-arc-spatial-source-{RUN_ID}'
SOURCE_DIR.mkdir(parents=True)
run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
run_command('add origin', ['git', 'remote', 'add', 'origin', REPOSITORY_URL], cwd=SOURCE_DIR)

github_token = os.environ.get('GITHUB_TOKEN', '').strip()
if not github_token:
    raise RuntimeError('Attach a Modal Secret containing GITHUB_TOKEN')
credential = base64.b64encode(
    f'x-access-token:{github_token}'.encode('utf-8')
).decode('ascii')
git_environment = os.environ.copy()
git_environment.pop('GITHUB_TOKEN', None)
git_environment.update({
    'GIT_TERMINAL_PROMPT': '0',
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
})
try:
    run_command(
        'fetch source',
        ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
        cwd=SOURCE_DIR, env=git_environment,
    )
    run_command('checkout source', ['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=SOURCE_DIR)
finally:
    git_environment.clear()
    del credential
RESOLVED_SOURCE_COMMIT = run_command(
    'resolve source', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR
)

PINNED_PACKAGES = {
    'accelerate': '1.12.0', 'diffusers': '0.39.0', 'einops': '0.8.1',
    'Pillow': '11.3.0', 'safetensors': '0.8.0', 'torch': '2.13.0',
    'tqdm': '4.67.1', 'transformers': '4.56.1',
}
def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
requirements = [
    f'{name}=={version}' for name, version in PINNED_PACKAGES.items()
    if installed_version(name) != version
]
if requirements:
    run_command(
        'install pinned dependencies',
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', '--no-cache-dir', *requirements],
    )
run_command(
    'install Diffusion-ARC source',
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input',
     '--no-deps', '--no-build-isolation', '-e', str(SOURCE_DIR)],
)
source_path = str((SOURCE_DIR / 'src').resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
test_environment = os.environ.copy()
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_environment['PYTHONPATH'] = source_path
test_started = time.perf_counter()
test_process = subprocess.run(
    [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
    cwd=SOURCE_DIR, env=test_environment, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, check=False,
)
print('\n'.join(test_process.stdout.splitlines()[-30:]))
if test_process.returncode != 0:
    raise RuntimeError(f'Unit tests returned {test_process.returncode}')
print('Resolved source commit:', RESOLVED_SOURCE_COMMIT)
print(f'Unit tests: PASS in {time.perf_counter() - test_started:.1f}s')


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
............................................................................................................................../usr/local/lib/python3.12/site-packages/torch/nn/functional.py:3006: UserWarning: Mismatch dtype between input and weight: input dtype = c10::BFloat16, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/layer_norm.cpp:346.)
  return torch.rms_norm(input, normalized_shape, weight, eps)
..................................
----------------------------------------------------------------------
Ran 160 tests in 3.505s

OK
Resolved source commit: a2f33795cd64ca63514e8ebc9f9b79daee7c5e06
Unit tests

## 3. Run both independent variants

The runner performs all preflights, one batched NVARC preparation, both VAE
target encodings, the complete compact run, checkpoint upload, a fresh FLUX
reload, and then the padded run. Progress bars report VAE preparation,
optimizer updates, loss, gradient norm, and peak GPU memory.


In [10]:
runner = SOURCE_DIR / 'scripts' / 'experiments' / 'run_single_task_spatial_ablation.py'
if not runner.is_file():
    raise RuntimeError(
        'Selected source revision lacks the spatial ablation runner; '
        'push the implementation and select that commit.'
    )
command = [
    sys.executable, '-u', str(runner),
    '--volume-root', str(VOLUME_ROOT), '--volume-name', VOLUME_NAME,
    '--run-id', RUN_ID, '--source-commit', RESOLVED_SOURCE_COMMIT,
]
experiment_environment = os.environ.copy()
experiment_environment['PYTHONDONTWRITEBYTECODE'] = '1'
experiment_environment['PYTHONPATH'] = source_path
process = subprocess.Popen(
    command, cwd=SOURCE_DIR, env=experiment_environment,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
if process.stdout is None:
    raise RuntimeError('Experiment process has no output stream')
for line in process.stdout:
    print(line, end='')
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(
        f'Spatial ablation exited with {returncode}; inspect {REPORT_PATH}'
    )
if not REPORT_PATH.is_file():
    raise RuntimeError(f'Experiment completed without report: {REPORT_PATH}')
REPORT = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
if REPORT.get('status') != 'COMPLETE':
    raise RuntimeError(f'Experiment report is not COMPLETE: {REPORT.get("status")}')
REPORT['comparison']


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!

Loading checkpoint shards: 100%|█████████████████████████████████████████████████| 2/2 [00:19<00:00,  9.53s/it]

VAE encode compact-3x3: 100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.78batch/s]

VAE encode compact-3x3: 100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00, 33.23batch/s]

VAE encode padded-30x30: 100%|██████████████████████████████████████████████| 32/32 [00:03<00:00,  8.40batch/s]

VAE encode padded-30x30: 100%|████████████████████████████████████████████████| 2/2 [00:00<00:00,  7.65batch/s]
compact-3x3 step 0 {'seen': 0, 'unseen': 0, 'canonical': 0} probe {'correct': {'total

{'compact-3x3': {'canonical_exact_grid_pass2': 0,
  'correct_probe_loss_ratio': 0.3704123482589641,
  'final_conditioning_loss_ratio': 0.9895413142674907,
  'mean_step_seconds': 0.21501982393750207,
  'peak_allocated_gib': 73.62001752853394,
  'seen_exact_grid_pass2': 0,
  'seen_oracle_3x3_accuracy': 0.16666666977107525,
  'unseen_exact_grid_pass2': 0},
 'padded-30x30': {'canonical_exact_grid_pass2': 0,
  'correct_probe_loss_ratio': 0.46735022014193195,
  'final_conditioning_loss_ratio': 0.9971472317200034,
  'mean_step_seconds': 0.5622832275468745,
  'peak_allocated_gib': 95.85143613815308,
  'seen_exact_grid_pass2': 0,
  'seen_oracle_3x3_accuracy': 0.15277778077870607,
  'unseen_exact_grid_pass2': 0}}

## 4. Checkpoint and result locations


In [11]:
checkpoint_summary = {
    name: {
        'status': payload['status'],
        'checkpoint': payload['saved_checkpoint']['path'],
        'remote_file_sizes_verified': payload['saved_checkpoint']['remote_file_sizes_verified'],
    }
    for name, payload in REPORT['variants'].items()
}
{'report': str(REPORT_PATH), 'comparison': REPORT['comparison'], 'checkpoints': checkpoint_summary}


{'report': '/mnt/diffusion-arc-models/results/single-task-spatial-overfitting-ablation/20260825T203819809220Z/report.json',
 'comparison': {'compact-3x3': {'canonical_exact_grid_pass2': 0,
   'correct_probe_loss_ratio': 0.3704123482589641,
   'final_conditioning_loss_ratio': 0.9895413142674907,
   'mean_step_seconds': 0.21501982393750207,
   'peak_allocated_gib': 73.62001752853394,
   'seen_exact_grid_pass2': 0,
   'seen_oracle_3x3_accuracy': 0.16666666977107525,
   'unseen_exact_grid_pass2': 0},
  'padded-30x30': {'canonical_exact_grid_pass2': 0,
   'correct_probe_loss_ratio': 0.46735022014193195,
   'final_conditioning_loss_ratio': 0.9971472317200034,
   'mean_step_seconds': 0.5622832275468745,
   'peak_allocated_gib': 95.85143613815308,
   'seen_exact_grid_pass2': 0,
   'seen_oracle_3x3_accuracy': 0.15277778077870607,
   'unseen_exact_grid_pass2': 0}},
 'checkpoints': {'compact-3x3': {'status': 'COMPLETE',
   'checkpoint': '/mnt/diffusion-arc-models/results/single-task-spatial-overf

## 5. Optional automatic shutdown

Shutdown is scheduled only after both uploaded checkpoint directories have
matching remote file sizes and the durable report is complete. Set
`DIFFUSION_ARC_AUTO_SHUTDOWN=0` before starting to keep the kernel alive.


In [12]:
durable_report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
if durable_report.get('status') != 'COMPLETE':
    raise RuntimeError('Refusing shutdown: report is not COMPLETE')
for variant_name in ('compact-3x3', 'padded-30x30'):
    checkpoint = durable_report['variants'][variant_name].get('saved_checkpoint')
    if not isinstance(checkpoint, dict) or not checkpoint.get('remote_file_sizes_verified'):
        raise RuntimeError(f'Refusing shutdown: {variant_name} checkpoint is unverified')
if not AUTO_SHUTDOWN:
    print('Automatic shutdown disabled; both checkpoints are durable.')
else:
    modal_task_id = os.environ.get('MODAL_TASK_ID', '').strip()
    if not modal_task_id:
        raise RuntimeError('MODAL_TASK_ID is unavailable; stop the kernel manually')
    shutdown_code = (
        'import subprocess, sys, time; ' 'time.sleep(10); '
        "subprocess.run(['modal', 'container', 'stop', '--yes', sys.argv[1]], check=False)"
    )
    subprocess.Popen(
        [sys.executable, '-c', shutdown_code, modal_task_id],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True,
    )
    print('Both checkpoints verified. Modal shutdown scheduled in 10 seconds.')


Both checkpoints verified. Modal shutdown scheduled in 10 seconds.


## Interpretation after the run

Use exact-grid pass@2 as the primary result. A compact-only success points
toward padding/loss imbalance. Success on seen but not unseen views points
toward presentation memorization. Similar correct and shuffled conditioning
behavior indicates that FLUX is ignoring NVARC context. Failure of both
variants shifts attention toward flow parameterization, timestep sampling,
or inference rather than padding alone.
